# Lab 11: Embeddings and Vector Search
This notebook demonstrates the semantic search system implemented for the Real Estate Market Monitor, satisfying all requirements of Lab 11.

In [3]:
import sys
!{sys.executable} -m pip install sentence-transformers chromadb torch pandas numpy

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

  Using cached sentence_transformers-5.5.0-py3-none-any.whl.metadata (18 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached transformers-5.8.1-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.14.0-py3-none-any.whl.metadata (14 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.

In [4]:
import sys
import os
import pandas as pd
import numpy as np

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from src.embeddings.embedder import Embedder, get_similarity_scores
from src.embeddings.chroma_store import ChromaStore
from src.embeddings.search_engine import SearchEngine
from src.embeddings.hybrid_search import hybrid_search

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Embeddings Module
Demonstrating embedding generation and similarity calculations.

In [5]:
embedder = Embedder()
sample_texts = [
    "Luxury penthouse with a view of the skyline",
    "High-end apartment overlooking the city",
    "Cozy rural cottage near the mountains"
]

embeddings = embedder.generate_embeddings(sample_texts)
print(f"Embedding shape: {embeddings.shape}")

scores = get_similarity_scores(embeddings[0], embeddings[1])
print(f"\nSimilarity between 'Luxury penthouse' and 'High-end apartment':")
for measure, score in scores.items():
    print(f"- {measure}: {score:.4f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6319.86it/s]


Model 'all-MiniLM-L6-v2' loaded successfully.
Embedding shape: (3, 384)

Similarity between 'Luxury penthouse' and 'High-end apartment':
- cosine_similarity: 0.5223
- dot_product: 0.5223
- euclidean_distance: 0.9774


## 2. ChromaDB Integration
Setting up and populating the vector database.

In [6]:
df = pd.read_csv('../data/processed/cleaned/cleaned_data.csv')
store = ChromaStore(path="../data/embeddings/chroma_db")

# Only populate if empty to save time
if store.count() == 0:
    documents = []
    metadatas = []
    ids = []
    
    for idx, row in df.iterrows():
        doc = embedder.combine_property_fields(row)
        meta = {
            "title": str(row.get('title', 'Unknown')),
            "type": str(row.get('type', 'Commercial')),
            "listing_id": str(row.get('listing_id', idx))
        }
        documents.append(doc)
        metadatas.append(meta)
        ids.append(str(idx))
    
    store.add_properties(documents, metadatas, ids)

print(f"Total documents in ChromaDB: {store.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4107.61it/s]


Model 'all-MiniLM-L6-v2' loaded successfully.
ChromaDB collection 'properties' initialized.
Total documents in ChromaDB: 237


## 3. Search System Demonstration
Comparing Keyword, Semantic, and Hybrid search.

In [7]:
searcher = SearchEngine(store, df, embedder)
query = "modern office space with parking"

semantic_res, keyword_res = searcher.compare_search(query)

hybrid_res = hybrid_search(semantic_res, keyword_res)
print(f"\n[Hybrid Search Results] (Top 3)")
for res in hybrid_res[:3]:
    print(f"- {res['data'].get('title', res['data'].get('metadata', {}).get('title'))} (Score: {res['score']:.4f})")


--- Comparing Search Methods for Query: 'modern office space with parking' ---

[Semantic Search Results] (Found: 5)
- property listing 364 (Dist: 0.5325)
- property listing 400 (Dist: 0.5350)
- property listing 3954 (Dist: 0.5355)
- property listing 374 (Dist: 0.5360)
- property listing 320 (Dist: 0.5394)

[Keyword Search Results] (Found: 0)

[Hybrid Search Results] (Top 3)
- property listing 364 (Score: 0.0164)
- property listing 400 (Score: 0.0161)
- property listing 3954 (Score: 0.0159)


## 4. Metadata Filtering
Demonstrating advanced filtering in vector search.

In [8]:
print("Search: 'warehouse' filtered by type: 'Industrial'")
filtered_results = store.query_semantic("warehouse", where={"type": "Industrial"})
for doc, meta in zip(filtered_results['documents'][0], filtered_results['metadatas'][0]):
    print(f"- {meta['title']} | Type: {meta['type']}")

Search: 'warehouse' filtered by type: 'Industrial'
- property listing 377 | Type: Industrial
- property listing 362 | Type: Industrial
- property listing 337 | Type: Industrial
- property listing 4427 | Type: Industrial
- property listing 4437 | Type: Industrial
